In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

*Install Libraries*

In [2]:
!pip install -U transformers sentence-transformers pymupdf arxiv faiss-cpu accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.0 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=c384263515764d3ffb0772bd743d648d6e1fa2bbb82e3ed70e73a80adfea478a
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninsta

*Import Libraries*

In [3]:
import arxiv
import fitz
import torch
import faiss
import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer

*Download Research Paper*

In [4]:
def download_paper(arxiv_url):

    paper_id = arxiv_url.split("/")[-1]

    search = arxiv.Search(id_list=[paper_id])
    paper = next(search.results())

    paper.download_pdf(filename="paper.pdf")

    print("Paper downloaded successfully!")

*Example later:*

In [5]:
url = "https://arxiv.org/abs/1706.03762"

download_paper(url)

/tmp/ipykernel_24/368246299.py:6: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  paper = next(search.results())


Paper downloaded successfully!


*Extract PDF Text*

In [6]:
def extract_text(pdf_path):

    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    return text

*Chunk Paper Into Sections*

In [7]:
def split_text(text, chunk_size=500):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks

*Load Embedding model*

In [8]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

*Create Vector Database*

In [9]:
def build_vector_store(chunks):

    embeddings = embedding_model.encode(chunks)

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(np.array(embeddings))

    return index, embeddings

*Semantic Search Function*

In [10]:
def search_paper(query, chunks, index):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(query_embedding, k=3)

    results = [chunks[i] for i in indices[0]]

    return results

*Load AI Summarization Model*

In [11]:
model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

*Summarization Function*

In [12]:
def summarize_text(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=200,
        min_length=80
    )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

*Generate Research Ideas*

In [13]:
def generate_research_ideas(summary):

    ideas = f"""
Based on the research summary below:

{summary}

Possible Research Directions:

1. Improve the model architecture
2. Train on a larger dataset
3. Compare with state-of-the-art models
4. Optimize computational efficiency
5. Explore new applications

Suggested Experiments:

• Reproduce baseline results  
• Evaluate performance on benchmark datasets  
• Modify model parameters  
• Analyze failure cases
"""

    return ideas

In [14]:
import fitz

def extract_text(pdf_path):

    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    return text

*Main Autonomous AI Assistant*

In [15]:
print("Text split into chunks")

# ensure build_vector_store exists
try:
    index, embeddings = build_vector_store(chunks)
except NameError:
    from sentence_transformers import SentenceTransformer
    import numpy as np
    import faiss

    embedding_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    def build_vector_store(chunks):
        embeddings = embedding_model.encode(chunks)
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(np.array(embeddings))
        return index, embeddings

    index, embeddings = build_vector_store(chunks)

print("Vector database created")

Text split into chunks


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NameError: name 'chunks' is not defined

*Paper Summary*

In [ ]:
paper_summary = summarize_text(text[:3000])

print("\n--- PAPER SUMMARY ---\n")

print(paper_summary)

*Ask Questions About Paper*

In [ ]:
question = input("\nAsk a question about the paper: ")

results = search_paper(question, chunks, index)

context = " ".join(results)

answer = summarize_text(context)

print("\n--- AI ANSWER ---\n")

print(answer)

*Generate Research Ideas*

In [ ]:
ideas = generate_research_ideas(paper_summary)

print("\n--- FUTURE RESEARCH IDEAS ---\n")

print(ideas)